In [2]:
import pickle, os
import numpy as np
from scipy import stats

class RestrictedUnpickler(pickle.Unpickler):
    SAFE_MODULES = {'numpy', 'numpy.core.multiarray', 'numpy.core', '_codecs', 'builtins', 'collections'}
    def find_class(self, module, name):
        if any(module.startswith(m) for m in self.SAFE_MODULES):
            return super().find_class(module, name)
        class Placeholder:
            def __init__(self, *a, **kw): pass
            def __setstate__(self, state): self.__dict__.update(state) if isinstance(state, dict) else None
        return Placeholder

BASE = r'D:\2026-1\Deep Learning\finalProject\Project\GitHubProject\COMP3242-Group-Project-DQN\results\ablations'
seeds = [0, 1, 2, 3, 4]

def load_seed_finals(subdir, values):
    """Returns {val: [final_avg_reward per seed]}"""
    data = {}
    for val in values:
        seed_finals = []
        for seed in seeds:
            path = os.path.join(BASE, subdir, f'{val}_seed_{seed}', 'results.pkl')
            if not os.path.exists(path):
                continue
            with open(path, 'rb') as f:
                d = RestrictedUnpickler(f).load()
            seed_finals.append(d['final_avg_reward'])
        if seed_finals:
            data[val] = seed_finals
    return data

ablations = {
    'Epsilon decay': ('epsilon_decay', [0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99, 0.995, 0.999]),
    'Learning rate': ('learning_rate', ['5e-05', '1e-04', '3e-04', '5e-04', '7e-04', '8e-04', '1e-03', '2e-03', '3e-03']),
    'Num layers':    ('num_layers',    [1, 2, 3, 4, 5]),
}

for ablation_name, (subdir, values) in ablations.items():
    print(f"\n{'='*55}")
    print(f" {ablation_name}")
    print(f"{'='*55}")
    print(f"  {'Value':<10} {'Mean':>8} {'Std':>8} {'95% CI':>20}")
    print(f"  {'-'*50}")

    data = load_seed_finals(subdir, values)

    for val, finals in data.items():
        arr = np.array(finals)
        mean, std = arr.mean(), arr.std()
        # 95% confidence interval (t-distribution, n=5)
        ci = stats.t.interval(0.95, df=len(arr)-1, loc=mean, scale=stats.sem(arr))
        print(f"  {str(val):<10} {mean:>8.1f} {std:>8.1f}   [{ci[0]:>6.1f}, {ci[1]:>6.1f}]")

    # Kruskal-Wallis test: are differences significant across all values?
    groups = list(data.values())
    if len(groups) >= 2:
        stat, p = stats.kruskal(*groups)
        sig = '*** (p<0.001)' if p < 0.001 else '** (p<0.01)' if p < 0.01 else '* (p<0.05)' if p < 0.05 else 'not significant'
        print(f"\n  Kruskal-Wallis test: H={stat:.2f}, p={p:.4f} → {sig}")

    # Pairwise: best config vs baseline (0.995 / 3e-04 / 3 layers)
    best_val = max(data, key=lambda v: np.mean(data[v]))
    baseline = {
        'epsilon_decay': 0.995,
        'learning_rate': '3e-04',
        'num_layers': 3
    }.get(subdir)

    if baseline and baseline in data and best_val != baseline:
        stat2, p2 = stats.mannwhitneyu(data[best_val], data[baseline], alternative='two-sided')
        sig2 = '*** (p<0.001)' if p2 < 0.001 else '** (p<0.01)' if p2 < 0.01 else '* (p<0.05)' if p2 < 0.05 else 'not significant'
        print(f"  Best ({best_val}) vs baseline ({baseline}): p={p2:.4f} → {sig2}")


 Epsilon decay
  Value          Mean      Std               95% CI
  --------------------------------------------------
  0.45          -48.1     27.7   [ -86.6,   -9.6]
  0.5           -32.4     67.5   [-126.1,   61.3]
  0.55          -38.1     25.1   [ -73.0,   -3.3]
  0.6             7.9     43.9   [ -53.0,   68.8]
  0.65          -34.1     38.0   [ -86.8,   18.6]
  0.7           -28.3     56.6   [-106.9,   50.3]
  0.75            7.3     68.8   [ -88.3,  102.9]
  0.8            29.9     70.9   [ -68.5,  128.4]
  0.85          -20.9     72.4   [-121.4,   79.7]
  0.9            14.1     18.0   [ -10.9,   39.1]
  0.95           42.9     70.2   [ -54.6,  140.3]
  0.99           -8.2     55.2   [ -84.9,   68.4]
  0.995         -25.8     46.5   [ -90.3,   38.8]
  0.999         -90.8      6.2   [ -99.4,  -82.2]

  Kruskal-Wallis test: H=21.57, p=0.0624 → not significant
  Best (0.95) vs baseline (0.995): p=0.2222 → not significant

 Learning rate
  Value          Mean      Std           

While the ablation study reveals trends suggesting that epsilon decay has the largest effect on performance (Kruskal-Wallis H=21.57, p=0.062), followed by learning rate (H=14.40, p=0.072) and network depth (H=4.62, p=0.329), none of the differences reach conventional statistical significance (p<0.05). This is consistent with the high variance observed across seeds in LunarLander-v3, where a single unlucky seed can produce rewards more than 70 points below the mean. To establish statistical significance, a larger seed set (≥10) or longer training runs would be required. We therefore report these results as empirical trends rather than definitive conclusions.